# AISKG v3.2.0 external BioRED/BioREDirect benchmark

This repository-native Google Colab notebook reproduces the external relation-classification experiment added in AISKG v3.2.0.

**Reporting boundaries**

- Every external system receives gold entity mentions and normalized identifiers; this is relation classification, not end-to-end NER plus RE.
- `AISKGRuleTransfer` and `AISKGConstrainedTransfer` are transfer adapters, not the unchanged mushroom-domain extractor.
- Sentence-local evaluation is primary; full-document evaluation is a cross-sentence stress test.
- Thresholds are selected on development data only, no test tuning is permitted, and the BC8 test set is locked after this experiment.
- Official BioRED/BioREDirect data, code, and model weights are downloaded from NCBI at runtime and are not redistributed by AISKG.

Choose **Runtime → Change runtime type → GPU** before running all cells.

In [ ]:
#@title 1. Configuration
from pathlib import Path

REPOSITORY_URL = "https://github.com/romenmeitei/AISKG.git"
REPOSITORY_REF = "v3.2.0"  # use release/v3.2.0 only while testing the pre-release branch
RUN_BIOREDIRECT = True
RUN_OFFLINE_SMOKE_TEST = True
SPLIT_SCHEME = "bioredirect_bc8_official"
BIOREDIRECT_REVISION = "main"
BIOREDIRECT_BATCH_SIZE = 8
BOOTSTRAP_ITERATIONS = 5000
GLOBAL_SEED = 20260826
FORCE_REDOWNLOAD = False
CLEAN_OUTPUT = True
USE_GOOGLE_DRIVE_FOR_WORK_CACHE = False

CONTENT_ROOT = Path("/content")
REPO_ROOT = CONTENT_ROOT / "AISKG"
RESULTS_DIR = CONTENT_ROOT / "AISKG_External_RE_Benchmark_Results_v1_0"
PUBLIC_RESULTS_DIR = CONTENT_ROOT / "AISKG_External_RE_Benchmark_Public_Results_v1_0_0"
WORK_DIR = CONTENT_ROOT / "AISKG_External_RE_Benchmark_Work"
print({"ref": REPOSITORY_REF, "seed": GLOBAL_SEED, "bootstraps": BOOTSTRAP_ITERATIONS})

In [ ]:
#@title 2. Clone the versioned AISKG repository
import shutil, subprocess

if USE_GOOGLE_DRIVE_FOR_WORK_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/AISKG_External_RE_Benchmark_Cache/work")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.check_call(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(REPO_ROOT)])
subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPOSITORY_REF])
resolved = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Resolved AISKG commit:", resolved)

In [ ]:
#@title 3. Install AISKG and analysis dependencies
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[external-re]"])
print("AISKG installed from", REPO_ROOT)

In [ ]:
#@title 4. Verify repository and run offline smoke test
import subprocess, sys
subprocess.check_call([sys.executable, str(REPO_ROOT / "verify_repository.py")], cwd=REPO_ROOT)
subprocess.check_call([sys.executable, str(REPO_ROOT / "scripts" / "verify_v3_2_0_release.py")], cwd=REPO_ROOT)
if RUN_OFFLINE_SMOKE_TEST:
    subprocess.check_call([sys.executable, str(REPO_ROOT / "scripts" / "run_external_re_smoke.py")], cwd=REPO_ROOT)

In [ ]:
#@title 5. GPU audit
import shutil, subprocess
if RUN_BIOREDIRECT and not shutil.which("nvidia-smi"):
    raise RuntimeError("Select a GPU runtime and restart from cell 1.")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)

## Full manuscript-grade run

The next cell downloads official NCBI assets, records their hashes and the resolved BioREDirect commit, runs official GPU inference, fits the prespecified transfer adapters, evaluates both scopes, and creates both a complete author-side result archive and a public-safe result archive.

In [ ]:
#@title 6. Run the external benchmark
import os, subprocess, sys
command = [
    sys.executable, str(REPO_ROOT / "scripts" / "run_external_re_benchmark.py"),
    "--package-root", str(REPO_ROOT),
    "--work-dir", str(WORK_DIR),
    "--output-dir", str(RESULTS_DIR),
    "--public-output-dir", str(PUBLIC_RESULTS_DIR),
    "--config", str(REPO_ROOT / "configs" / "external_re" / "benchmark_config.json"),
    "--rules", str(REPO_ROOT / "ontology" / "external_re" / "biored_trigger_rules.json"),
    "--split-scheme", SPLIT_SCHEME,
    "--bioredirect-revision", BIOREDIRECT_REVISION,
    "--batch-size", str(BIOREDIRECT_BATCH_SIZE),
    "--bootstrap-iterations", str(BOOTSTRAP_ITERATIONS),
    "--seed", str(GLOBAL_SEED),
    "--modes", "sentence_local", "full_document",
]
if CLEAN_OUTPUT: command.append("--clean")
if FORCE_REDOWNLOAD: command.append("--force-download")
command.append("--run-bioredirect" if RUN_BIOREDIRECT else "--skip-bioredirect")
print("Executing:\n", " ".join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
for line in process.stdout: print(line, end="")
if process.wait() != 0: raise RuntimeError("External benchmark failed")

In [ ]:
#@title 7. Inspect quality gates and metrics
import json, pandas as pd
from IPython.display import display
quality = json.loads((RESULTS_DIR / "QUALITY_GATES.json").read_text())
print(json.dumps(quality, indent=2))
if quality.get("status") != "PASS": raise RuntimeError("Do not report this run")
metrics = pd.read_csv(RESULTS_DIR / "system_metrics.csv")
comparisons = pd.read_csv(RESULTS_DIR / "paired_comparisons.csv")
display(metrics)
display(comparisons)

In [ ]:
#@title 8. Display figures
from IPython.display import Image, display
for name in ["figure_external_relation_f1.png", "figure_per_relation_recall_full_document.png"]:
    display(Image(filename=str(RESULTS_DIR / name)))

In [ ]:
#@title 9. Download full and public-safe result archives
from google.colab import files
FULL_ZIP = RESULTS_DIR.with_suffix(".zip")
PUBLIC_ZIP = PUBLIC_RESULTS_DIR.with_suffix(".zip")
for archive in [FULL_ZIP, PUBLIC_ZIP]:
    if not archive.is_file(): raise FileNotFoundError(archive)
    print("Downloading", archive.name)
    files.download(str(archive))

## Interpretation

A manuscript-grade run does not require AISKG to outperform BioREDirect. It must provide a transparent estimate of portability. Do not tune any feature or threshold after inspecting BC8 test results; use a newly designated untouched test set for subsequent model development.